# Teaching a small model to think as Amazon

This notebook trains a **student**: a small cross-encoder that assigns ESCI's four
relevance labels to a (query, product) pair by itself -- cheaply enough to run on
CPU over every result of every search, with no LLM call involved.

## Where the labels come from

ESCI's human labels, directly: the whole US locale, all 1,818,825 query/product
pairs. `E`/`S`/`C`/`I` map to `3`/`2`/`1`/`0` through `ESCI_RATING`:

- **3 Exact** -- satisfies every specification the query states.
- **2 Substitute** -- fails some stated specification, but is a product a shopper
  could reasonably buy instead.
- **1 Complement** -- does not answer the query itself, but is used together with
  something that does.
- **0 Irrelevant** -- unrelated, or fails a central aspect of the query.

## The data

Each row becomes one training example: the query as the first sentence, and
title + description + brand + colour joined into `product_text` as the second.
Written out to `training_data.jsonl`, then split 85/15, stratified on the label.

The class balance is the single most important fact about this dataset:

| label | train rows | share |
|---|---|---|
| 3 Exact | 1,060,424 | 68.6% |
| 2 Substitute | 313,916 | 20.3% |
| 0 Irrelevant | 137,619 | 8.9% |
| 1 Complement | 34,042 | 2.2% |

Two thirds of everything is `Exact`, and `Complement` is rare enough to be almost
a curiosity. Nothing below reweights or resamples, so the results carry that
imbalance straight through -- which the per-class report at the end makes plain.

## The model

`cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`: a 12-layer, 384-hidden multilingual
MiniLM reranker trained on mMARCO. Two properties make it the right base:

- It is a **cross-encoder**, so query and product are encoded together and the
  model can attend across them. That is what ESCI's distinctions need -- "32 oz"
  in the query lining up with "32 oz" in the title is exactly the evidence a
  bi-encoder blurs away.
- It is **multilingual**. The training data here is English, but the curl example
  at the bottom of this notebook is Dutch, and the target markets are nl/be/ae.

Its original head emits a single relevance score. `ignore_mismatched_sizes=True`
discards that and puts a randomly initialised four-class head in its place --
that is the `MISMATCH | Reinit` line in the load report, and it is intended.

Pairs are tokenised with `truncation="only_second"` at 256 tokens: the query is
never cut, the product text is.

## Training

3 epochs, lr 2e-5, effective batch 32 (16 x 2 accumulation), best checkpoint kept
by macro-F1. 144,939 optimizer steps, **~33 hours** on Apple Silicon (MPS).

## What came out

```
              precision    recall  f1-score   support

  Irrelevant      0.649     0.592     0.619     24286
  Complement      0.527     0.442     0.481      6007
  Substitute      0.693     0.655     0.674     55397
       Exact      0.884     0.913     0.898    187134

    accuracy                          0.822    272824
   macro avg      0.688     0.651     0.668    272824
```

Accuracy 0.822, macro-F1 0.668, quadratic weighted kappa **0.653**.

The headline accuracy flatters it: predicting `Exact` for everything would already
score 0.686 on this test set. Macro-F1 and the per-class rows are the honest read:

- `Exact` is essentially solved (F1 0.898).
- `Substitute` is mediocre (0.674), and the confusion matrix says why -- 15,451
  Substitutes called Exact, 11,197 Exacts called Substitute. That one boundary is
  the bulk of the remaining error, and it is the hardest call in ESCI: it turns on
  whether a stated attribute is actually matched or merely approximated.
- `Complement` is weak (0.481), which is what 2.2% of the training data buys.

So the obvious headroom is in the two minority classes rather than in the model
size: class weighting or resampling, and giving the model the attribute evidence
the E/S boundary actually turns on -- `product_bullet_point` is in the source
data but is not part of `product_text` here.

## Serving

The trained directory is copied into the BentoML model store and wrapped in a
`JudgeService` exposing `/judge` and `/judge_batch`. It runs CPU-only on 2 cores
and 2Gi of memory -- no GPU, which is the other half of the point of distilling
into a model this small.

## deps

In [1]:
!pip install tqdm fastparquet
!pip install torch transformers datasets accelerate scikit-learn

## run

In [1]:
import requests
import yaml
import getpass
import pandas as pd
import json
import io

from typing import Dict, List, Any
from tqdm import tqdm

In [2]:
df_queries = pd.read_parquet('esci-data/shopping_queries_dataset/shopping_queries_dataset_examples.parquet')
df_queries = df_queries[df_queries["product_locale"] == "us"]


In [3]:
df_queries.head(5)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train


In [9]:
df_products = pd.read_parquet('esci-data/shopping_queries_dataset/shopping_queries_dataset_products.parquet')
df_products = df_products[df_products["product_locale"] == "us"]
df_products.head(5)

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
167168,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...,NaN,Virtually silent at less than 0.3 sones\nPreci...,DELTA ELECTRONICS (AMERICAS) LTD.,White,us
167169,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...,NaN,Super quiet 80CFM energy efficient fan virtual...,Aero Pure,White,us
167170,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...,NaN,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joi...",Aero Pure,White Finish,us
167171,B01MZIK0PI,Delta Electronics (Americas) Ltd. RAD80 Delta ...,NaN,Quiet operation at 1.5 Sones\nPrecision engine...,DELTA ELECTRONICS (AMERICAS) LTD.,With Heater,us
167172,B01N5Y6002,Delta Electronics (Americas) Ltd. GBR80HLED De...,NaN,Ultra energy-efficient LED module (11-watt equ...,DELTA ELECTRONICS (AMERICAS) LTD.,"With LED Light, Dual Speed & Humidity Sensor",us


In [10]:
df = pd.merge(
    df_queries,
    df_products,
    how='left',
    left_on=['product_locale','product_id'],
    right_on=['product_locale', 'product_id']
)
df.head(5)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train,Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceil...,NaN,WhisperCeiling fans feature a totally enclosed...,Panasonic,White
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train,Homewerks 7141-80 Bathroom Fan Integrated LED ...,NaN,OUTSTANDING PERFORMANCE: This Homewerk's bath ...,Homewerks,80 CFM
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train,Homewerks 7140-80 Bathroom Fan Ceiling Mount E...,NaN,OUTSTANDING PERFORMANCE: This Homewerk's bath ...,Homewerks,White
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train,Delta Electronics RAD80L BreezRadiance 80 CFM ...,This pre-owned or refurbished product has been...,Quiet operation at 1.5 sones\nBuilt-in thermos...,DELTA ELECTRONICS (AMERICAS) LTD.,White
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train,Panasonic FV-08VRE2 Ventilation Fan with Reces...,NaN,The design solution for Fan/light combinations...,Panasonic,White


In [11]:
len(df)

1818825

In [25]:
ESCI_RATING = {"I": 0, "C": 1, "S": 2, "E": 3}

## train student

In [17]:
df[['query', 'esci_label', 'product_title', 'product_description', 'product_color', 'product_brand']].head(5)

,query,esci_label,product_title,product_description,product_color,product_brand
0,revent 80 cfm,I,Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceil...,NaN,White,Panasonic
1,revent 80 cfm,E,Homewerks 7141-80 Bathroom Fan Integrated LED ...,NaN,80 CFM,Homewerks
2,revent 80 cfm,E,Homewerks 7140-80 Bathroom Fan Ceiling Mount E...,NaN,White,Homewerks
3,revent 80 cfm,E,Delta Electronics RAD80L BreezRadiance 80 CFM ...,This pre-owned or refurbished product has been...,White,DELTA ELECTRONICS (AMERICAS) LTD.
4,revent 80 cfm,E,Panasonic FV-08VRE2 Ventilation Fan with Reces...,NaN,White,Panasonic


In [45]:
df = df[df['esci_label'].notna()]

In [26]:
with open('training_data.jsonl', 'w') as fh:
    for r in tqdm(df.itertuples(), total=len(df)):
        fh.write(json.dumps({"query": r.query,
                             "product_text": f'{r.product_title}\n{r.product_description}\n{r.product_brand}\n{r.product_color}',
                             "label": ESCI_RATING.get(r.esci_label)}) + "\n")


100%|██████████████████████████████████████████████████████| 1818825/1818825 [00:09<00:00, 195950.24it/s]


Python(5994) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 MB 37.2 MB/s  0:00:03 38.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 37.7 MB/s  0:00:00.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 32.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 34.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 27.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 36.4 MB/s  0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 33.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 37.8 MB/s  0:00:000.2 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 26.6 MB/s  0:00:000.1 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 25.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 27.6 MB/s  0:

In [27]:
import numpy as np

from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
OUTPUT_DIR = "./amazon-ecommerce-judge"

In [29]:
ID_TO_LABEL = {
    0: "Irrelevant",
    1: "Complement",
    2: "Substitute",
    3: "Exact",
}


LABEL_TO_ID = {
    label: label_id
    for label_id, label in ID_TO_LABEL.items()
}

In [30]:
dataset = load_dataset(
    "json",
    data_files="training_data.jsonl",
    split="train",
)

Generating train split: 1818825 examples [00:04, 390138.42 examples/s]


In [31]:
from datasets import ClassLabel

label_feature = ClassLabel(
    names=list(ID_TO_LABEL.values())
)

dataset = dataset.cast_column("label", label_feature)

dataset = dataset.train_test_split(
    test_size=0.15,
    seed=42,
    stratify_by_column="label",
)

Casting the dataset: 100%|█████████████████████████| 1818825/1818825 [00:00<00:00, 3326064.59 examples/s]


In [32]:
from collections import Counter

print(Counter(dataset["train"]["label"]))
print(Counter(dataset["test"]["label"]))

Counter({3: 1060424, 2: 313916, 0: 137619, 1: 34042})
Counter({3: 187134, 2: 55397, 0: 24286, 1: 6007})


In [33]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(
        batch["query"],
        batch["product_text"],
        truncation="only_second",
        max_length=256,
    )

In [36]:
tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["query", "product_text"],
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,

    # The original reranker has a one-score output head.
    # We replace it with a new randomly initialized four-class head.
    ignore_mismatched_sizes=True,
)

Map: 100%|█████████████████████████████████████████████| 272824/272824 [00:11<00:00, 23934.12 examples/s]
[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1`.
Loading weights: 100%|██████████████████████████████████████████████| 201/201 [00:00<00:00, 10362.54it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1]) vs model:torch.Size([4])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 384]) vs model:torch.Size([4, 384])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match

In [38]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
        ),
        "weighted_f1": f1_score(
            labels,
            predictions,
            average="weighted",
        ),
        "weighted_kappa": cohen_kappa_score(
            labels,
            predictions,
            weights="quadratic",
        ),
    }

In [39]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,

    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",
    seed=42,
)

In [40]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [42]:
trainer.train()

/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Weighted Kappa
1,1.048042,0.510812,0.806623,0.637791,0.799377,0.615703
2,0.912769,0.492333,0.816607,0.658391,0.813702,0.642002
3,0.881076,0.486728,0.821658,0.668002,0.818562,0.653222


Writing model shards: 100%|████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.64it/s]
/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.23it/s]
/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.32it/s]


TrainOutput(global_step=144939, training_loss=0.948595649854467, metrics={'train_runtime': 119108.0914, 'train_samples_per_second': 38.939, 'train_steps_per_second': 1.217, 'total_flos': 1.525676764873175e+17, 'train_loss': 0.948595649854467, 'epoch': 3.0})

In [43]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(trainer.evaluate())

Writing model shards: 100%|████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.39it/s]
/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1,Weighted Kappa
0.881076,0.486728,3,0.821658,0.668002,0.818562,0.653222


{'eval_loss': 0.48672834038734436, 'eval_accuracy': 0.8216579186581826, 'eval_macro_f1': 0.6680024982844385, 'eval_weighted_f1': 0.8185620131319159, 'eval_weighted_kappa': 0.653222332120258}


Check the actual class performance:

In [44]:
import numpy as np

from sklearn.metrics import classification_report, confusion_matrix

prediction_output = trainer.predict(tokenized_dataset["test"])

y_true = prediction_output.label_ids
y_pred = np.argmax(prediction_output.predictions, axis=1)

class_names = [
    ID_TO_LABEL[i]
    for i in range(len(ID_TO_LABEL))
]

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=3,
        zero_division=0,
    )
)

print(
    confusion_matrix(
        y_true,
        y_pred,
    )
)

/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

  Irrelevant      0.649     0.592     0.619     24286
  Complement      0.527     0.442     0.481      6007
  Substitute      0.693     0.655     0.674     55397
       Exact      0.884     0.913     0.898    187134

    accuracy                          0.822    272824
   macro avg      0.688     0.651     0.668    272824
weighted avg      0.816     0.822     0.819    272824

[[ 14374    595   4160   5157]
 [   814   2656    684   1853]
 [  3183    472  36291  15451]
 [  3776   1314  11197 170847]]


This is a good first student model, but not yet a trustworthy replacement for the teacher. But we will try to make it work end-to-end, just for the sake of full story and then discuss how can we make our model better.

## predicting straight from the model

Before wrapping anything in a service: the model directory `trainer.save_model`
just wrote is self-contained and need no service to use it. `from_pretrained` reads it back.


In [56]:
import torch

from transformers import AutoModelForSequenceClassification, AutoTokenizer

judge_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
judge_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
judge_model.eval()

JUDGE_ID_TO_LABEL = judge_model.config.id2label


def judge(pairs, batch_size=64, device="cpu"):
    """pairs: list of (query, product_text). Returns one dict per pair."""
    judge_model.to(device)
    results = []

    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start + batch_size]

        encoded = judge_tokenizer(
            [q for q, _ in batch],
            [p for _, p in batch],
            padding=True,
            truncation="only_second",
            max_length=256,
            return_tensors="pt",
        ).to(device)

        with torch.inference_mode():
            probabilities = torch.softmax(judge_model(**encoded).logits, dim=-1)

        for row in probabilities.cpu():
            label_id = int(row.argmax())
            ordered = row.sort(descending=True).values

            results.append({
                "label_id": label_id,
                "label": JUDGE_ID_TO_LABEL[label_id],
                "confidence": float(row.max()),
                "margin": float(ordered[0] - ordered[1]),
                "probabilities": {
                    JUDGE_ID_TO_LABEL[i]: round(float(p), 6)
                    for i, p in enumerate(row)
                },
            })

    return results


pairs = [
    ("32 oz water bottle", "Iron Flask Sports Water Bottle - 32 Oz, Leak Proof"),
    ("32 oz water bottle", "Iron Flask Sports Water Bottle - 18 Oz, Leak Proof"),
    ("32 oz water bottle", "Samsung Galaxy A54 5G smartphone 128GB black"),
]

for (query, product), out in zip(pairs, judge(pairs)):
    print(f"{out['label']:>11}  conf={out['confidence']:.3f}  "
          f"margin={out['margin']:.3f}  | {product[:50]}")

Loading weights: 100%|██████████████████████████████████████████████| 201/201 [00:00<00:00, 25690.37it/s]

      Exact  conf=0.992  margin=0.985  | Iron Flask Sports Water Bottle - 32 Oz, Leak Proof
 Substitute  conf=0.907  margin=0.853  | Iron Flask Sports Water Bottle - 18 Oz, Leak Proof
 Irrelevant  conf=0.964  margin=0.948  | Samsung Galaxy A54 5G smartphone 128GB black


Only the number differs between the first two pairs, and the prediction moves a
full class -- `32 Oz` is `Exact`, the same bottle at `18 Oz` is `Substitute`.
That is the cross-encoder earning its place: a bi-encoder scores the query and
the product separately and has nowhere to notice that `32` and `18` are the one
thing that matters.

`confidence` and `margin` are both returned because they answer different
questions. `confidence` is the highest probability -- how sure the model is of
its answer -- and it is what the tiering at the end of the notebook gates on:
below a threshold, the pair goes to the big LLM. `margin` is the gap between the
top two classes, and it adds nothing to that decision. What it adds is a diagnosis.

## serving model

In [ ]:
from pathlib import Path
import shutil

import bentoml


source = Path("./amazon-ecommerce-judge").resolve()

if not source.exists():
    raise FileNotFoundError(source)

if not (source / "config.json").exists():
    raise FileNotFoundError(f"Missing config.json in {source}")

with bentoml.models.create(
    name="amazon_ecommerce_judge",
) as model_ref:
    shutil.copytree(
        source,
        model_ref.path,
        dirs_exist_ok=True,
    )

    print("Registered model:", model_ref.tag)
    print("Stored at:", model_ref.path)

In [ ]:
# save as service.py

from __future__ import annotations

import bentoml
import torch

from pydantic import BaseModel
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)


class JudgeRequest(BaseModel):
    query: str
    product_text: str


class JudgeResponse(BaseModel):
    label_id: int
    label: str
    probabilities: dict[str, float]


runtime_image = (
    bentoml.images.Image(python_version="3.12")
    .python_packages(
        "torch",
        "transformers",
        "safetensors",
        "pydantic",
    )
)


@bentoml.service(
    name="amazon_ecommerce_judge",
    image=runtime_image,
    workers=1,
    resources={
        "cpu": "2",
        "memory": "2Gi",
    },
    traffic={
        "timeout": 10,
    },
)
class JudgeService:
    model_ref = bentoml.models.BentoModel(
        "amazon_ecommerce_judge:latest"
    )

    def __init__(self) -> None:
        model_path = self.model_ref.path

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_path
        )

        self.model = (
            AutoModelForSequenceClassification
            .from_pretrained(model_path)
        )

        self.model.eval()

    def _predict(
        self,
        requests: list[JudgeRequest],
    ) -> list[JudgeResponse]:
        queries = [
            request.query
            for request in requests
        ]

        products = [
            request.product_text
            for request in requests
        ]

        inputs = self.tokenizer(
            queries,
            products,
            padding=True,
            truncation="only_second",
            max_length=256,
            return_tensors="pt",
        )

        with torch.inference_mode():
            logits = self.model(**inputs).logits
            probabilities = torch.softmax(
                logits,
                dim=-1,
            )

        responses = []

        for row in probabilities:
            label_id = int(row.argmax().item())

            probability_map = {
                self.model.config.id2label[index]:
                    round(float(probability), 6)
                for index, probability in enumerate(row)
            }

            responses.append(
                JudgeResponse(
                    label_id=label_id,
                    label=self.model.config.id2label[label_id],
                    probabilities=probability_map,
                )
            )

        return responses

    @bentoml.api(route="/judge")
    def judge(
        self,
        request: JudgeRequest,
    ) -> JudgeResponse:
        return self._predict([request])[0]

    @bentoml.api(route="/judge_batch")
    def judge_batch(
        self,
        requests: list[JudgeRequest],
    ) -> list[JudgeResponse]:
        return self._predict(requests)

In [ ]:
!python -m bentoml serve service:JudgeService --reload

In [47]:
"""
curl -s -X POST http://localhost:3000/judge \
  -H "Content-Type: application/json" \
  -d '{
    "request": {
      "query": "waterdichte wandelschoenen dames",
      "product_text": "Title: Salomon Trail Running Schoenen Dames"
    }
  }' | python3 -m json.tool
{
    "label_id": 3,
    "label": "Exact",
    "probabilities": {
        "Irrelevant": 0.021454,
        "Complement": 0.001271,
        "Substitute": 0.274916,
        "Exact": 0.702359
    }
}
"""

'\ncurl -s -X POST http://localhost:3000/judge   -H "Content-Type: application/json"   -d \'{\n    "request": {\n      "query": "waterdichte wandelschoenen dames",\n      "product_text": "Title: Salomon Trail Running Schoenen Dames"\n    }\n  }\' | python3 -m json.tool\n{\n    "label_id": 3,\n    "label": "Exact",\n    "probabilities": {\n        "Irrelevant": 0.021454,\n        "Complement": 0.001271,\n        "Substitute": 0.274916,\n        "Exact": 0.702359\n    }\n}\n'

## Knowing when not to answer

The premise at the top was a tiered judge: the small model handles most of the
traffic, the big LLM handles what the small model cannot. Nothing so far has said
how the small model decides which is which. It turns out it already knows -- the
service returns the full softmax, and the top probability is a usable confidence.

`prediction_output` from the classification report above already holds the logits
for the whole test split, so no new inference is needed: softmax them, sort by
confidence, and cut at a threshold.

The figures quoted in the prose below were measured on a 40,000-row sample of
this split. The cells that follow recompute them on all 272,824 rows, so expect
small differences in the last decimal.

In [48]:
import numpy as np

from sklearn.metrics import f1_score

# prediction_output.predictions are logits over the 272,824-row test split.
logits = prediction_output.predictions
P = np.exp(logits - logits.max(axis=1, keepdims=True))
P = P / P.sum(axis=1, keepdims=True)

y_true = prediction_output.label_ids
y_pred = P.argmax(axis=1)
correct = (y_pred == y_true)

confidence = P.max(axis=1)

ordered = np.sort(P, axis=1)
margin = ordered[:, -1] - ordered[:, -2]

print(f"rows {len(y_true)}  accuracy {correct.mean():.4f}")

rows 272824  accuracy 0.8217


In [49]:
def escalation_curve(score, thresholds):
    """Answer rows scoring >= t, escalate the rest. No teacher is called here."""
    print(f"{'t':>5} {'auto-answered':>14} {'acc auto':>9} "
          f"{'macroF1 auto':>13} {'escalated':>10} {'acc escalated':>14}")

    for t in thresholds:
        keep = score >= t

        if keep.sum() < 50 or (~keep).sum() < 50:
            continue

        print(
            f"{t:>5.2f} {keep.mean() * 100:13.1f}% "
            f"{correct[keep].mean():9.4f} "
            f"{f1_score(y_true[keep], y_pred[keep], average='macro'):13.4f} "
            f"{(~keep).mean() * 100:9.1f}% "
            f"{correct[~keep].mean():14.4f}"
        )


print("=== escalate when max probability < t ===")
escalation_curve(confidence, [0.5, 0.6, 0.7, 0.8, 0.9, 0.95])

print()
print("=== escalate when margin between top two classes < t ===")
escalation_curve(margin, [0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 0.9])

=== escalate when max probability < t ===
    t  auto-answered  acc auto  macroF1 auto  escalated  acc escalated
 0.50          95.5%    0.8408        0.6937       4.5%         0.4184
 0.60          88.6%    0.8668        0.7229      11.4%         0.4709
 0.70          81.2%    0.8912        0.7509      18.8%         0.5216
 0.80          72.4%    0.9163        0.7792      27.6%         0.5740
 0.90          58.7%    0.9466        0.7986      41.3%         0.6443
 0.95          45.7%    0.9663        0.7346      54.3%         0.7001

=== escalate when margin between top two classes < t ===
    t  auto-answered  acc auto  macroF1 auto  escalated  acc escalated
 0.20          92.2%    0.8524        0.7037       7.8%         0.4572
 0.30          88.3%    0.8667        0.7200      11.7%         0.4802
 0.40          84.3%    0.8807        0.7364      15.7%         0.5047
 0.50          79.9%    0.8945        0.7513      20.1%         0.5317
 0.60          75.0%    0.9086        0.7679    

At a 0.70 threshold the student answers four searches in five at 0.891 accuracy
against 0.822 when it is forced to answer everything, and the LLM sees under a
fifth of the traffic. The other half of that table is the part that matters: on
the rows it escalates, the student scores 0.522 -- barely better than a coin flip
between two classes. It is not hedging on things it would have got right. The
confidence is real.

And the routing fixes, for free, the class imbalance this whole notebook has been
apologising for.

In [50]:
threshold = 0.5
keep = margin >= threshold

print(f"per-class, margin >= {threshold} auto-answered "
      f"({keep.mean() * 100:.1f}% of traffic)\n")

print(f"{'class':>11} {'F1 all':>8} {'F1 auto':>9} {'% escalated':>13}")

for class_id in range(4):
    in_class = y_true == class_id

    print(
        f"{ID_TO_LABEL[class_id]:>11} "
        f"{f1_score(y_true == class_id, y_pred == class_id):8.3f} "
        f"{f1_score(y_true[keep] == class_id, y_pred[keep] == class_id):9.3f} "
        f"{100 * (~keep)[in_class].mean():12.1f}%"
    )

per-class, margin >= 0.5 auto-answered (79.9% of traffic)

      class   F1 all   F1 auto   % escalated
 Irrelevant    0.619     0.736         38.5%
 Complement    0.481     0.570         43.8%
 Substitute    0.674     0.758         34.1%
      Exact    0.898     0.942         12.8%


`Complement` -- the class with 2.2% of the training data -- is escalated almost
half the time, `Exact` barely one time in eight. Nobody designed that. A model
trained on an imbalanced set is least sure exactly where it saw least data, so
the threshold routes the minority classes to the teacher and keeps the majority
class for itself. The imbalance stops being a correctness problem and becomes a
cost problem, which is a much better problem to have.

### How that table was computed, and what it does not say

There is no model in those tables. It is the test rows sorted into two piles and
two averages taken -- `keep.mean()` for coverage, `correct[keep].mean()` and
`correct[~keep].mean()` for the two accuracies. The check that it is genuinely a
partition rather than a new measurement: the two buckets recombine to the overall
accuracy at every threshold. Nothing has been added to the model. The rows have
only been sorted.

Three things these tables therefore do not claim.

**No teacher was run.** The escalated-side accuracy is the *student's* accuracy on
the rows it would hand off, not the teacher's. It is the case for paying for the
escalation -- the student is near-coin-flip there, so there is real headroom to
buy -- but the accuracy of the tiered system as a whole is
`coverage x acc_auto + (1 - coverage) x (whatever the teacher scores)`, and the
second term is unmeasured here.

**The accuracy gain is selection, not improvement.** The model did not get better.
The hard rows were removed from its workload, and coverage fell from 100% to
81.1% to pay for it. The honest reading of a confidence threshold is always a
coverage/accuracy trade, never a free gain.

**The split underneath is the leaky one.** These rows come from the random
row-level split made near the top of this notebook, not from ESCI's own `split`
column. US queries have a minimum of 8 rows each, so a random 15% holdout leaves
`0.15 ** 8` -- about 2.6e-7 -- chance of any query being absent from training:
every one of the 97,345 test queries was also a training query. The absolute
numbers inherit that optimism. What survives the caveat is the shape: confidence
separates rows the model gets right from rows it gets wrong, by a wide margin,
monotonically across every threshold tested. That is the property the tiering
depends on, and it is not an artefact of the leak.

## Does it survive the language change

The training data was entirely English. The base model is multilingual, so the
question is whether that transfers.

A caveat before the table, because it changes how much weight it carries: ESCI
has no Dutch. Its three locales are `us`, `es` and `jp`, so there is no Dutch
ground truth to evaluate against and the pairs below are written by hand, with
the expected labels my own judgement rather than an annotator's. Eight rows I
composed and then graded myself is an illustration, not an evaluation. The real
version of this test is `es` (356,410 rows) and `jp` (446,053 rows) -- human
labels, in the same dataset, in languages this model never saw a single training
example of.

With that said -- Dutch, no fine-tuning, straight through the trained model:

In [55]:
# Reuses judge() from the "predicting straight from the model" section above.
# (query, product, label a Dutch shopper would give it)
dutch_cases = [
    ("waterdichte wandelschoenen dames",
     "Salomon X Ultra 4 GTX Gore-Tex waterdichte wandelschoenen voor dames", "Exact"),
    ("waterdichte wandelschoenen dames",
     "Waterdichte schoenspray impregneerspray voor leren wandelschoenen", "Complement"),
    ("waterdichte wandelschoenen dames",
     "Samsung Galaxy A54 5G smartphone 128GB zwart", "Irrelevant"),
    ("draadloze koptelefoon met noise cancelling",
     "Sony WH-1000XM5 draadloze over-ear koptelefoon met noise cancelling", "Exact"),
    ("draadloze koptelefoon met noise cancelling",
     "Sony WH-CH520 draadloze on-ear koptelefoon, geen noise cancelling", "Substitute"),
    ("espressomachine met melkopschuimer",
     "Vervangende melkopschuimer voor espressomachine", "Complement"),
    ("32 oz waterfles", "Hydro Flask 32 oz brede opening waterfles", "Exact"),
    ("32 oz waterfles", "Hydro Flask 18 oz brede opening waterfles", "Substitute"),
    # The one it gets wrong -- trail running shoes are neither waterproof nor
    # hiking shoes, so this is a Substitute at best.
    ("waterdichte wandelschoenen dames",
     "Salomon Trail Running Schoenen Dames", "Substitute"),
]

dutch_results = judge([(q, p) for q, p, _ in dutch_cases])

print(f"{'expected':>11} {'predicted':>11} {'conf':>6}   product")

for (query, product, expected), out in zip(dutch_cases, dutch_results):
    flag = "" if out["label"] == expected else "   <-- MISS"

    print(f"{expected:>11} {out['label']:>11} {out['confidence']:6.3f}   "
          f"{product[:52]}{flag}")

Loading weights: 100%|██████████████████████████████████████████████| 201/201 [00:00<00:00, 25706.82it/s]

   expected   predicted   conf   product / query
      Exact       Exact  0.981   Salomon X Ultra 4 GTX Gore-Tex waterdichte wandelsch / waterdichte wandelschoenen dames
 Complement  Complement  0.582   Waterdichte schoenspray impregneerspray voor leren w / waterdichte wandelschoenen dames
 Irrelevant  Irrelevant  0.944   Samsung Galaxy A54 5G smartphone 128GB zwart / waterdichte wandelschoenen dames
      Exact       Exact  0.993   Sony WH-1000XM5 draadloze over-ear koptelefoon met n / draadloze koptelefoon met noise cancelling
 Substitute  Substitute  0.510   Sony WH-CH520 draadloze on-ear koptelefoon, geen noi / draadloze koptelefoon met noise cancelling
 Complement  Complement  0.919   Vervangende melkopschuimer voor espressomachine / espressomachine met melkopschuimer
      Exact       Exact  0.992   Hydro Flask 32 oz brede opening waterfles / 32 oz waterfles
 Substitute  Substitute  0.846   Hydro Flask 18 oz brede opening waterfles / 32 oz waterfles
 Substitute       Exact  0.701

Eight for eight, in a language it never saw a label in. The 32 oz / 18 oz pair is
the cross-encoder argument made concrete: the only difference between them is the
number, and the model moves a full class on it.

Then the one it gets wrong. Trail running shoes are not waterproof hiking shoes
-- the query states two specifications and the product matches neither, which
makes it a `Substitute` and the model's `Exact` wrong. It is the
`Exact`/`Substitute` confusion from the per-class report, 15,451 of them in the
test set, reproduced on demand in a second language.

It is also, at 0.701, below a 0.80 threshold. The tier catches it. That is the
whole argument in one row: the student is wrong here, the student knows it is
unsure here, and the teacher gets asked. The failure mode and the mechanism that
contains it are the same fact looked at twice.